# Developing chicken heart

This notebook is a guide to the `chicken_heart` workflow. It shows how raw
data are mapped into the fields expected by CytoBridge, how a new model run is
passed to downstream analysis, and which later steps start from files saved
for the paper instead of the new run. Edit the paths in **Setup** before
starting.

For a small example that runs on generated data, see [Synthetic
preprocessing](../data_preparation/synthetic_preprocessing.ipynb). The [data and
checkpoint guide](../../data_checkpoints.md) lists inputs distributed outside
the package.

## Setup

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

from CytoBridge.workflow import (
    WorkflowOptions,
    build_workflow_plan,
    load_workflow_config,
    render_workflow_plan,
    run_workflow,
)
DATASET_CONFIG = 'chicken_heart'
RAW_H5AD = Path("data/chicken_heart_raw.h5ad")
OUTPUT_DIR = Path("tutorial_outputs/chicken_heart")
PREPROCESS_ONLY_DIR = Path("tutorial_outputs/chicken_heart_preprocess_only")
DOWNSTREAM_RERUN_DIR = Path("tutorial_outputs/chicken_heart_downstream_rerun")
ALIGNED_H5AD = OUTPUT_DIR / "preprocess" / 'chicken_heart_aligned.h5ad'
MODEL_DIR = OUTPUT_DIR / "training"


import CytoBridge as cb

RAW_10X_DIR = Path("data/GSE149457_RAW")
METADATA_H5AD = Path("data/chicken_heart_spatial_merged_with_meta.h5ad")
REFERENCE_ALIGNMENT_H5AD = Path("data/heart_aligned_all_timepoints.h5ad")
PREPARATION_DIR = RAW_H5AD.parent / "chicken_heart_preparation"
RUN_RAW_DATA_ASSEMBLY = False


RUN_TRAINING = False
RUN_PREPROCESS_ONLY = False
RUN_DOWNSTREAM = False

In [2]:
config, config_source = load_workflow_config(DATASET_CONFIG)
dataset = config["dataset"]
scientific = config["scientific"]
downstream = config["downstream"]
preprocess = config["preprocess"]
align = preprocess["align"]

spatial_obs_keys = align.get("spatial_obs_keys")
if spatial_obs_keys:
    spatial_source = ", ".join(f"obs[{key!r}]" for key in spatial_obs_keys)
else:
    spatial_source = f"obsm[{align.get('input_spatial_key', 'spatial')!r}]"

pd.DataFrame(
    {
        "setting": [
            "dataset",
            "configuration",
            "raw time column",
            "raw annotation column",
            "aligned annotation column",
            "raw count layer",
            "raw spatial coordinates",
            "aligned spatial coordinates",
            "model time values",
            "classifier neighbors",
        ],
        "value": [
            dataset["display_name"],
            config_source,
            preprocess["time_key"],
            preprocess["annotation_source"],
            dataset["annotation_key"],
            align.get("expression_layer", "X"),
            spatial_source,
            f"obsm[{dataset['spatial_key']!r}]",
            ", ".join(map(str, align["time_mapping"].values())),
            scientific["classifier_k"],
        ],
    }
)

,setting,value
0,dataset,Developing chicken heart
1,configuration,example configuration: chicken_heart
2,raw time column,timepoint
3,raw annotation column,celltype_prediction
4,aligned annotation column,celltype_prediction
5,raw count layer,counts
6,raw spatial coordinates,obsm['spatial_ot_input']
7,aligned spatial coordinates,obsm['spatial_aligned']
8,model time values,"0.0, 1.0, 2.0, 3.0"
9,classifier neighbors,1


## Start a new model run

The dataset configuration records the count layer, time mapping, spatial
coordinates, alignment settings, and model settings. Start here when fitting a
new model. The command reads the raw H5AD, writes the aligned H5AD, fits the
ligand--receptor edge predictor when the model uses one, trains CytoBridge, and
runs the analyses selected in the configuration. No separate preprocessing
command is needed first.

### Assemble the chicken-heart H5AD

The public GSE149457 10x matrices contain the raw counts and spot coordinates.
Two paper-retained H5AD files are also required: `METADATA_H5AD` supplies the
spot roster, region labels, and cell-type labels, and
`REFERENCE_ALIGNMENT_H5AD` supplies the matching row order used by the original
paper preparation. These two H5AD files are not generated by the public 10x
download or by the standard workflow.

The first call joins the raw counts to those retained annotations. The second
call starts from the raw coordinates, records them as `spatial_original`, and
writes `spatial_ot_input`; for D7 this applies the recorded 180-degree
pre-orientation before CytoBridge fits a new alignment.

**Input:** `RAW_10X_DIR`, `METADATA_H5AD`, and `REFERENCE_ALIGNMENT_H5AD`

**Output:** `RAW_H5AD`, containing counts, annotations, `spatial_original`, and
`spatial_ot_input`

**Continue with:** the **Run a new dataset from raw counts** command below

In [3]:
if RUN_RAW_DATA_ASSEMBLY:
    PREPARATION_DIR.mkdir(parents=True, exist_ok=True)
    reference_input = PREPARATION_DIR / "chicken_heart_reference_input.h5ad"
    cb.pp.prepare_chicken_heart_input(
        raw_dir=RAW_10X_DIR,
        metadata_h5ad=METADATA_H5AD,
        aligned_reference_h5ad=REFERENCE_ALIGNMENT_H5AD,
        output_h5ad=reference_input,
        output_table=PREPARATION_DIR / "model_input.csv",
        manifest_path=PREPARATION_DIR / "preparation.json",
        graph_database=cb.pp.bundled_graph_database_path(DATASET_CONFIG),
        repair_legacy_d7_left_right=False,
    )
    cb.pp.prepare_chicken_heart_ot_input(
        input_h5ad=reference_input,
        output_h5ad=RAW_H5AD,
        output_table=PREPARATION_DIR / "chicken_heart_ot_input.csv",
        manifest_path=PREPARATION_DIR / "ot_input.json",
    )
else:
    print(
        "Raw-data assembly is off. Add the public 10x matrices and the two "
        "paper-retained H5AD files, then set RUN_RAW_DATA_ASSEMBLY = True."
    )

Raw-data assembly is off. Add the public 10x matrices and the two paper-retained H5AD files, then set RUN_RAW_DATA_ASSEMBLY = True.


### Run a new dataset from raw counts

```bash
cytobridge workflow --config chicken_heart --train \
  --input-h5ad <raw.h5ad> --output-dir <run> --device cuda
```

**Input:** raw H5AD, dataset configuration, and the included or user-supplied LR database

**Output:** `<run>`/preprocess/chicken_heart_aligned.h5ad; `<run>`/preprocess/edge_classifier/chicken_heart_edge_model.pt when the configuration uses a ligand--receptor edge predictor; `<run>`/training/`<stage>`/best_model.pth or score_model.pth; `<run>`/training/adata.h5ad; `<run>`/training/training_history.csv; `<run>`/training/training_run_summary.json; `<run>`/downstream/summary.json, result tables, and figures

**Continue with:** inspect `<run>`/downstream, or rerun only the downstream analysis in a new output directory

Start here with raw data. This one command preprocesses, trains, and runs the analyses selected in the configuration. The separate preprocessing-only run below is optional.

The next two cells show the same operation through the Python API. Leave
`RUN_TRAINING = False` when reading the documentation; set it to `True` only
after the paths above point to your data. The compact table shows the file
used by each step; the complete package plan is stored in `training_plan_text` if you
want to print it in Jupyter.

In [4]:
training_options = WorkflowOptions(
    input_h5ad=RAW_H5AD,
    output_dir=OUTPUT_DIR,
    train=True,
)
training_plan = build_workflow_plan(
    config,
    source=config_source,
    options=training_options,
)
training_plan_text = render_workflow_plan(training_plan)
pd.DataFrame(
    [
        {
            "step": "preprocess",
            "input": RAW_H5AD,
            "output": ALIGNED_H5AD,
        },
        {
            "step": "train",
            "input": ALIGNED_H5AD,
            "output": MODEL_DIR,
        },
        {
            "step": "downstream",
            "input": f"{ALIGNED_H5AD} + {MODEL_DIR}",
            "output": OUTPUT_DIR / "downstream",
        },
    ]
)

,step,input,output
0,preprocess,data/chicken_heart_raw.h5ad,tutorial_outputs/chicken_heart/preprocess/chic...
1,train,tutorial_outputs/chicken_heart/preprocess/chic...,tutorial_outputs/chicken_heart/training
2,downstream,tutorial_outputs/chicken_heart/preprocess/chic...,tutorial_outputs/chicken_heart/downstream


In [5]:
if RUN_TRAINING:
    if not RAW_H5AD.is_file():
        raise FileNotFoundError(f"Update RAW_H5AD before training: {RAW_H5AD}")
    training_result = run_workflow(config, options=training_options)
    training_result
else:
    print("Training is off. Set RUN_TRAINING = True to start a new model run.")

Training is off. Set RUN_TRAINING = True to start a new model run.


## Inspect the aligned data without training (optional)

Use this separate command only when you want to examine the aligned H5AD before
committing to a model fit. It writes to `PREPROCESS_ONLY_DIR` and does not fit
an edge predictor or a CytoBridge model. It is not an earlier step in the model
run above; when you are ready to train, use the first command from the raw H5AD.

```bash
cytobridge workflow --config chicken_heart --step preprocess \
  --input-h5ad <raw.h5ad> --output-dir <preprocess-only>
```

**Input:** raw H5AD and the dataset configuration

**Output:** `<preprocess-only>`/preprocess/chicken_heart_aligned.h5ad and preprocessing records; no edge predictor or CytoBridge model

**Continue with:** inspect the aligned H5AD; use the first command, starting again from the raw H5AD, when ready to fit a model

This is an alternative inspection run, not a prerequisite for training.

In [6]:
preprocess_only_options = WorkflowOptions(
    input_h5ad=RAW_H5AD,
    output_dir=PREPROCESS_ONLY_DIR,
    steps=("preprocess",),
)
preprocess_only_plan = build_workflow_plan(
    config,
    source=config_source,
    options=preprocess_only_options,
)
preprocess_only_plan_text = render_workflow_plan(preprocess_only_plan)
pd.DataFrame(
    [
        {
            "step": "preprocess only",
            "input": RAW_H5AD,
            "output": PREPROCESS_ONLY_DIR / "preprocess" / ALIGNED_H5AD.name,
        }
    ]
)

,step,input,output
0,preprocess only,data/chicken_heart_raw.h5ad,tutorial_outputs/chicken_heart_preprocess_only...


In [7]:
if RUN_PREPROCESS_ONLY:
    if not RAW_H5AD.is_file():
        raise FileNotFoundError(f"Update RAW_H5AD before preprocessing: {RAW_H5AD}")
    preprocess_only_result = run_workflow(
        config,
        options=preprocess_only_options,
    )
    preprocess_only_result
else:
    print(
        "Preprocessing-only run is off. Set RUN_PREPROCESS_ONLY = True "
        "to write an aligned H5AD without training."
    )

Preprocessing-only run is off. Set RUN_PREPROCESS_ONLY = True to write an aligned H5AD without training.


## Run downstream analysis again (optional)

The first command already runs downstream analysis. Use this section only when
you want to repeat it from the same aligned H5AD and fitted model. The rerun
writes to `DOWNSTREAM_RERUN_DIR`, leaving the original results unchanged.

```bash
cytobridge workflow --config chicken_heart --step downstream \
  --aligned-h5ad <run>/preprocess/chicken_heart_aligned.h5ad \
  --model-dir <run>/training --output-dir <downstream-rerun>
```

**Input:** aligned H5AD; `<run>`/training; dataset-matched LR database

**Output:** `<downstream-rerun>`/downstream/summary.json; slice_data/*.h5ad; velocity/velocity_components.npz; growth/growth_by_cell.csv; composition/celltype_composition.csv; communication and ligand_receptor tables; standard figures

**Continue with:** paper-specific continuation shown in the paper-figure notebook

The first command already runs these analyses. Use this form only to repeat downstream analysis, and choose a new output directory so the original results are not overwritten.

In [8]:
downstream_options = WorkflowOptions(
    aligned_h5ad=ALIGNED_H5AD,
    model_dir=MODEL_DIR,
    output_dir=DOWNSTREAM_RERUN_DIR,
    steps=("downstream",),
)
downstream_plan = build_workflow_plan(
    config,
    source=config_source,
    options=downstream_options,
)
downstream_plan_text = render_workflow_plan(downstream_plan)
pd.DataFrame(
    [
        {
            "step": "downstream",
            "input": f"{ALIGNED_H5AD}; {MODEL_DIR}",
            "output": DOWNSTREAM_RERUN_DIR / "downstream",
        }
    ]
)

,step,input,output
0,downstream,tutorial_outputs/chicken_heart/preprocess/chic...,tutorial_outputs/chicken_heart_downstream_reru...


In [9]:
if RUN_DOWNSTREAM:
    missing = [path for path in (ALIGNED_H5AD, MODEL_DIR) if not path.exists()]
    if missing:
        raise FileNotFoundError(f"Missing aligned data or model directory: {missing}")
    downstream_result = run_workflow(config, options=downstream_options)
    downstream_result
else:
    print("Downstream rerun is off. Set RUN_DOWNSTREAM = True to repeat it.")

Downstream rerun is off. Set RUN_DOWNSTREAM = True to repeat it.


## Paper figures

The headings state exactly where each calculation starts:

- **Continue from the model run above** reads the `OUTPUT_DIR` created here.
- **Start from the paper's saved files** reads the tables, arrays, or models
  retained from the exact paper analysis. It does not read the current
  `OUTPUT_DIR` unless the step says so.
- **Required paper files not included** names an input or page builder that is
  not shipped in this repository; no command is shown in its place.

Commands beginning with `python scripts/...` or `python -m scripts...` must be
run from the root of a cloned source repository. Each step names its input,
output, and the notebook that continues from it.

- [LR-prior and interaction ablations](../paper_figures/interaction_ablation.ipynb)
- [Five-dataset benchmark](../paper_figures/loto_benchmark.ipynb)
- [Training histories](../paper_figures/training_histories.ipynb)

### Continue from the model run above: calculate the chicken-heart paper outputs (Main Figure 3)

```bash
python scripts/run_chicken_heart_paper_downstream.py \
  --run-root <run> \
  --input-h5ad <run>/preprocess/chicken_heart_aligned.h5ad \
  --model-dir <run>/training \
  --standard-downstream <run>/downstream \
  --output-dir <chicken-heart-paper-output> --device cuda
```

**Input:** aligned H5AD, retrained model, and its downstream directory

**Output:** perturbation, interaction-off, LR-time-course, and model interaction-score tables and figures

**Continue with:** assemble the selected Main Figure 3 panels

The calculation is available for a new run, but the repository does not currently contain the updated Main Figure 3 page-assembly command.

### Continue from the model run above: compare the saved coordinate systems (S7-S8)

```bash
python scripts/plot_chicken_heart_alignment.py \
  --input-h5ad <run>/preprocess/chicken_heart_aligned.h5ad \
  --output-dir <alignment-figure>
```

**Input:** aligned H5AD containing obsm['spatial_original'], obsm['spatial_ot_input'], obsm['spatial_aligned'], and uns['spatial_alignment_info']

**Output:** coordinate-comparison PDF/PNG, source CSV, caption, and provenance JSON

**Continue with:** compare with S7-S8 or use the saved coordinate table in a new layout

The standard workflow stores the alignment record inside the H5AD rather than in a separate JSON file. The exact S7-S8 page assembly is not included.

### Use the growth results from the model run above (S9)

File or directory to use (not a command): `<run>/downstream/growth`

**Input:** the growth directory written by the complete raw-data run

**Output:** `<run>`/downstream/growth/growth_by_cell.csv and growth_timepoint_grid.pdf

**Continue with:** use the table and standard plot in a new analysis, or compare them with S9

No second downstream command is required. The exact S9 page-assembly command is not included.

### Use the velocity results from the model run above (S10)

File or directory to use (not a command): `<run>/downstream/velocity`

**Input:** the velocity directory written by the complete raw-data run

**Output:** `<run>`/downstream/velocity/velocity_components.npz and full/drift/interaction vector PDFs

**Continue with:** use the arrays and standard plots in a new analysis, or compare them with S10

No second downstream command is required. The exact S10 page assembly and its comparison-method input are not included.

## Saved files

- Aligned data: `tutorial_outputs/chicken_heart/preprocess/chicken_heart_aligned.h5ad`
- Training directory: `tutorial_outputs/chicken_heart/training`
- Downstream directory: `tutorial_outputs/chicken_heart/downstream`